# Testing Our Own Findings: The Da Critique

**Computational Text Analysis Pack · Module 11 — From Counts to Argument**

*Technique:* Permutation tests, resampling, and robustness of a published-style claim

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/11c_testing_our_own_findings.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Nan Z. Da's "The Computational Case against Computational Literary Studies"
(2019) makes a blunt charge: that computational literary studies is full of
statistical errors, and that **"what is robust is obvious, and what is not
obvious is not robust."**

The responses — from Underwood, Piper, Klein and the statistician Taylor Arnold
— argue she misreads the statistics and the epistemology. Ben Schmidt called the
piece a magic trick. The survey says to present both sides, and the right way to
do that in a lab notebook is not to summarise the argument but to **apply it to
our own results.**

So: take a finding this course produced, and try to break it.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# A finding of exactly the kind this course generates: which words are
# characteristic of Nigerian-language news versus the rest.
nigerian = df[df["is_nigerian"]]
other = df[~df["is_nigerian"]]

def corpus_counts(frame):
    c = Counter()
    for t in frame["text"]:
        c.update(tokenise(t))
    return c, sum(c.values())

ca, na = corpus_counts(nigerian)
cb, nb_ = corpus_counts(other)

def log_ratio(w):
    ra = (ca[w] + 0.5) / na
    rb = (cb[w] + 0.5) / nb_
    return np.log2(ra / rb)

shared = [w for w in ca if ca[w] + cb[w] >= 50]
finding = pd.DataFrame({"word": shared,
                        "log_ratio": [log_ratio(w) for w in shared]})
finding = finding.sort_values("log_ratio", ascending=False)
print("THE CLAIM: these words are most characteristic of Nigerian-language news")
print(finding.head(10).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Test 1 — is it stable, or an artefact of these particular documents?

Da's robustness objection has a concrete form: if the finding depends on which
documents happened to be sampled, it is not a finding about the corpus. Bootstrap
the documents and see whether the top words stay on top.

In [ ]:
rng = np.random.default_rng(11)
TOP = 20
baseline = set(finding.head(TOP)["word"])
survival = Counter()

for _ in range(15):
    a = nigerian.sample(frac=1.0, replace=True, random_state=int(rng.integers(1e6)))
    b = other.sample(frac=1.0, replace=True, random_state=int(rng.integers(1e6)))
    ca2, na2 = corpus_counts(a)
    cb2, nb2 = corpus_counts(b)
    scores = {w: np.log2(((ca2[w] + .5) / na2) / ((cb2[w] + .5) / nb2))
              for w in shared}
    top = sorted(scores, key=scores.get, reverse=True)[:TOP]
    survival.update(top)

stable = [(w, survival[w]) for w in baseline]
stable.sort(key=lambda t: -t[1])
print(f"How often each original top-{TOP} word stayed in the top {TOP}, over 15 resamples:")
for w, n in stable:
    bar = "#" * n
    print(f"  {w:<18}{n:>3}/15  {bar}")

## Test 2 — how big is the effect against pure chance?

A permutation test discards the labels entirely: shuffle which documents count
as "Nigerian", recompute, and see what log-ratios appear *by construction*. Any
real effect has to stand clear of that null distribution.

In [ ]:
pooled = pd.concat([nigerian, other], ignore_index=True)
n_a = len(nigerian)
null_max = []
for i in range(12):
    perm = pooled.sample(frac=1.0, random_state=100 + i).reset_index(drop=True)
    ca3, na3 = corpus_counts(perm.iloc[:n_a])
    cb3, nb3 = corpus_counts(perm.iloc[n_a:])
    scores = [np.log2(((ca3[w] + .5) / na3) / ((cb3[w] + .5) / nb3))
              for w in shared]
    null_max.append(max(scores))

observed = finding["log_ratio"].iloc[0]
print(f"observed top log-ratio      : {observed:.2f}")
print(f"largest under label shuffling: {max(null_max):.2f} "
      f"(mean {np.mean(null_max):.2f})")
print()
if observed > max(null_max):
    print("The effect clears the permutation null. This one survives Da's test.")
else:
    print("The effect does NOT clear the null. It would not survive.")

plt.figure(figsize=(6.5, 3.5))
plt.hist(null_max, bins=8, alpha=.75, label="max under shuffled labels")
plt.axvline(observed, color="crimson", lw=2, label="observed")
plt.xlabel("log ratio")
plt.legend()
plt.tight_layout()

## What each side would say about this result

**Da's reading.** The surviving words are function words and orthographic
markers of the languages themselves. The result is robust *and* obvious — it
says Hausa articles are written in Hausa. Her charge is that a great deal of
computational literary criticism is this, dressed up.

**The respondents' reading.** Underwood and Piper argue that establishing a
baseline quantitatively is not a failure but a precondition, and that Da's
replications discard the interpretive framing that made the original claims
meaningful. Arnold's objection is technical: the statistics she calls errors are
often standard practice she has misread.

**What the notebook can settle, and what it cannot.** The tests above are
mechanical and either pass or fail. Whether a surviving result is *interesting*
is not a statistical question, and neither side claims it is.

### 🔧 Your turn 1

Run the same two tests on a keyness result from notebook 04a. Does it survive bootstrap and permutation as cleanly as this one?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Keyness results computed on smaller subcorpora — a single language, or one
category — are noticeably less stable: the top-20 survival counts drop into the
single digits for several words. That is the useful lesson. It is not that
keyness is unsound, it is that the stability of a keyness list depends on
subcorpus size, and no part of the standard workflow tells you that. Running
fifteen bootstrap resamples costs seconds and belongs in any paper that reports
a keyword list.

</details>